# Day 1: Embeddings & Chunking Practical

In this session, we will explore:
1.  **Chunking Strategies**: How to split text effectively.
2.  **Embeddings**: Generating vector representations of text using Ollama.
3.  **Semantic Similarity**: Comparing text meanings using cosine similarity.

## Prerequisites
Make sure you have Ollama installed and running (`ollama serve`).
Put the following libraries in your environment:
```bash
pip install litellm numpy scikit-learn
```

In [1]:
!pip install litellm numpy scikit-learn

  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached aiohttp-3.13.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached fastuuid-0.14.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (1.1 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached tiktoken-0.12.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached attrs-25.4.0-py3-non

In [4]:
import numpy as np
from litellm import embedding
from sklearn.metrics.pairwise import cosine_similarity

## 1. Chunking Strategies

Let's start with a sample text.

In [5]:
text = """
Retrieval-Augmented Generation (RAG) is a technique for enhancing the accuracy and reliability of generative AI models with facts fetched from external sources.
Building a RAG pipeline can be complex, but it's worth it for the improved performance on specific tasks.
RAG allows LLMs to access private data without retraining.
This is crucial for enterprise applications where data privacy is paramount.
"""

### Fixed-Size Chunking
Simple splitting by character count.

In [6]:
def fixed_size_chunking(text, chunk_size=100, overlap=20):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks

fixed_chunks = fixed_size_chunking(text)
print(f"Fixed-Size Chunks ({len(fixed_chunks)}):")
for i, chunk in enumerate(fixed_chunks):
    print(f"Chunk {i+1}: {chunk!r}")

Fixed-Size Chunks (6):
Chunk 1: '\nRetrieval-Augmented Generation (RAG) is a technique for enhancing the accuracy and reliability of g'
Chunk 2: 'and reliability of generative AI models with facts fetched from external sources.\nBuilding a RAG pip'
Chunk 3: ".\nBuilding a RAG pipeline can be complex, but it's worth it for the improved performance on specific"
Chunk 4: 'formance on specific tasks.\nRAG allows LLMs to access private data without retraining.\nThis is cruci'
Chunk 5: 'ining.\nThis is crucial for enterprise applications where data privacy is paramount.\n'
Chunk 6: 'nt.\n'


### Recursive Character Chunking (Concept)
Splitting by separators (paragraphs, then sentences, then words) to keep semantic meaning intact.
(We'll implement a simple version here).

In [7]:
def recursive_chunking(text, separators=["\n\n", "\n", ". ", " "], chunk_size=100):
    # This is a simplified version. Real implementations are more complex (e.g., LangChain's).
    # For now, let's just split by sentences.
    sentences = text.split('. ')
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= chunk_size:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

recursive_chunks = recursive_chunking(text)
print(f"\nRecursive/Sentence Chunks ({len(recursive_chunks)}):")
for i, chunk in enumerate(recursive_chunks):
    print(f"Chunk {i+1}: {chunk!r}")


Recursive/Sentence Chunks (2):
Chunk 1: ''
Chunk 2: "Retrieval-Augmented Generation (RAG) is a technique for enhancing the accuracy and reliability of generative AI models with facts fetched from external sources.\nBuilding a RAG pipeline can be complex, but it's worth it for the improved performance on specific tasks.\nRAG allows LLMs to access private data without retraining.\nThis is crucial for enterprise applications where data privacy is paramount.\n."


## 2. Embeddings with LiteLM & Ollama

Ensuring you have run `ollama pull nomic-embed-text` or similar embedding model.
Valid models: `all-minilm`, `nomic-embed-text`, `llama3` (can do embeddings too but slower).

In [8]:
def get_embedding(text, model="ollama/nomic-embed-text"):
    try:
        response = embedding(model=model, input=[text])
        return response['data'][0]['embedding']
    except Exception as e:
        print(f"Error getting embedding: {e}")
        return []

# Example: Get embedding for the word "cat"
vec_cat = get_embedding("cat")
print(f"\nEmbedding for 'cat' (first 10 dims): {vec_cat[:10]}")
print(f"Vector Dimension: {len(vec_cat)}")


Embedding for 'cat' (first 10 dims): [0.031655695, 0.062293977, -0.1354668, -0.038176645, 0.061799806, 0.035159986, -0.07183496, 0.023455463, -0.07575366, -0.011956119]
Vector Dimension: 768


In [9]:
vec_cat = get_embedding("Dog")
print(f"\nEmbedding for 'cat' (first 10 dims): {vec_cat[:10]}")
print(f"Vector Dimension: {len(vec_cat)}")


Embedding for 'cat' (first 10 dims): [-0.0013825342, 0.0020674132, -0.15709326, 0.009839326, 0.06675564, 0.011215101, -0.04012337, -0.013772016, -0.013270739, -0.060610402]
Vector Dimension: 768


## 3. Semantic Similarity / Search

Let's see how embeddings capture meaning.

In [10]:
# Define some sentences
sentences = [
    "The cat sits on the mat",      # 0
    "A feline rests on the rug",    # 1 (Semantically similar to 0)
    "I love eating pizza",          # 2 (Completely different)
    "The dog chases the ball",      # 3 (Related to animals, but different action)
]



# Generate embeddings for all
embeddings = [get_embedding(s) for s in sentences]



In [13]:
embeddings[1]

[0.043577053,
 0.033245742,
 -0.13916853,
 -0.068706594,
 0.047019005,
 0.005806624,
 -0.038656887,
 0.041292917,
 -0.06496792,
 -0.09524048,
 -0.012034259,
 0.05069572,
 0.02734091,
 0.028371895,
 0.016177073,
 -0.049354102,
 0.03643535,
 -0.02622668,
 0.0366638,
 -0.037520956,
 -0.058890093,
 0.02842746,
 -0.042554196,
 -0.025197031,
 0.082565784,
 0.04425026,
 0.04475204,
 0.04713132,
 0.054499906,
 0.039412413,
 0.022101963,
 0.007513601,
 -0.005359006,
 -0.049978055,
 0.013758008,
 0.007108908,
 0.03963549,
 0.038385432,
 0.066969834,
 0.00682303,
 -0.038442872,
 0.0676582,
 -0.0077557135,
 0.047086082,
 0.013471786,
 -0.01549091,
 0.042896975,
 -0.019337872,
 0.0022368587,
 0.006837618,
 -0.03556546,
 0.042589515,
 -0.0062207617,
 -0.04490235,
 0.048084877,
 0.021445718,
 0.048714932,
 -0.0077945753,
 -0.027834987,
 0.015418939,
 0.088023916,
 -0.011728257,
 -0.06969773,
 0.09604642,
 0.019894378,
 -0.05088655,
 -0.0048458427,
 0.00640297,
 0.024142535,
 -0.026583113,
 0.07852955

In [15]:
# Ensure we got valid embeddings
embeddings = [e for e in embeddings if e] 

if len(embeddings) == len(sentences):
    # Calculate Cosine Similarity Matrix
    # Similarity = 1 (Identical) to -1 (Opposite)
    # Usually 0 to 1 for text embeddings.
    sim_matrix = cosine_similarity(embeddings)

    print("\nCosine Similarity Matrix:")
    print(sim_matrix)

    print("\nSimilarity Analysis:")
    print(f"'{sentences[0]}' vs '{sentences[1]}': {sim_matrix[0][1]:.4f}")
    print(f"'{sentences[0]}' vs '{sentences[2]}': {sim_matrix[0][2]:.4f}")
else:
    print("Could not generate all embeddings. Make sure Ollama is running and model is pulled.")


Cosine Similarity Matrix:
[[1.         0.72394705 0.47198477 0.43797158]
 [0.72394705 1.         0.43394024 0.49842492]
 [0.47198477 0.43394024 1.         0.49146669]
 [0.43797158 0.49842492 0.49146669 1.        ]]

Similarity Analysis:
'The cat sits on the mat' vs 'A feline rests on the rug': 0.7239
'The cat sits on the mat' vs 'I love eating pizza': 0.4720
